# Autoencoders

In this notebook, we will explore autoencoder models. These are models in which inputs are *encoded* to an intermediate representation before being *decoded* to reconstruct the original inputs. Autoencoders use unsupervised training methods and are interesting both as models in their own right and as a method for pre-training useful representations for supervised tasks such as classification. Autoencoders as a pre-training method were covered in the additional material in the sixth lecture slides.

## Exercise 1: Linear Autoencoders

In this exercise, we will train a simple **contractive** autoencoder—one in which the hidden representation has a smaller dimension than the input. The objective is to minimise the mean squared error between the original inputs and the reconstructed outputs. To begin, we will use models where both the encoder and decoder are simple affine transformations.

When training an autoencoder, the target outputs are the original inputs themselves. A simple way to integrate this into our `mlp` framework is to define a new data provider that inherits from a base data provider (e.g., `MNISTDataProvider`) and overrides the `next` method to return the input batch as both inputs and targets. Such a data provider has been provided for you in `mlp.data_providers` as `MNISTAutoencoderDataProvider`.

---

### **Your Tasks:**

1. Use the `MNISTAutoencoderDataProvider` to train an autoencoder model with:
   - A **50-dimensional hidden representation**
   - Both encoder and decoder defined by **affine transformations**
   - A **sum of squared differences error**
   - A **basic gradient descent learning rule** with learning rate **0.01**
   
2. Initialize the biases to **zero** and use a **uniform Glorot initialization** for both layer weights.

3. Train the model for **25 epochs** with a **batch size of 50**.

In [ ]:
import numpy as np
import logging
import mlp.layers as layers
import mlp.models as models
import mlp.optimisers as optimisers
import mlp.errors as errors
import mlp.learning_rules as learning_rules
import mlp.data_providers as data_providers
import mlp.initialisers as initialisers
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
#TODO Create the model and train it

---

Using the function defined in the cell below (from the first lab notebook), plot a batch of the original images and the autoencoder reconstructions.

In [ ]:
def show_batch_of_images(img_batch, fig_size=(3, 3), num_rows=None):
    """Display a batch of images in a grid layout."""
    fig = plt.figure(figsize=fig_size)
    batch_size, im_height, im_width = img_batch.shape
    if num_rows is None:
        # Calculate grid dimensions to give square(ish) grid
        num_rows = int(batch_size**0.5)
    num_cols = int(batch_size * 1. / num_rows)
    if num_rows * num_cols < batch_size:
        num_cols += 1
    # Initialize empty array to tile image grid into
    tiled = np.zeros((im_height * num_rows, im_width * num_cols))
    # Iterate over images in batch and their indices
    for i, img in enumerate(img_batch):
        # Calculate grid row and column indices
        r, c = i % num_rows, i // num_rows
        tiled[r * im_height:(r + 1) * im_height, 
              c * im_width:(c + 1) * im_width] = img
    ax = fig.add_subplot(111)
    ax.imshow(tiled, cmap='Greys', vmin=0., vmax=1.)
    ax.axis('off')
    fig.tight_layout()
    plt.show()
    return fig, ax

In [ ]:
#TODO show the generated images and original images using the above function

---

### Optional Extension: Principal Component Analysis

*This section is provided for students who are also taking MLPR or are otherwise familiar with eigendecompositions and PCA. Feel free to skip this section if it doesn't apply to you.*

For a linear (affine) contractive autoencoder model trained with a sum of squared differences error function, there is an analytic solution for the optimal model parameters corresponding to [Principal Component Analysis (PCA)](https://en.wikipedia.org/wiki/Principal_component_analysis).

If we have a training dataset of $N$ $D$-dimensional vectors $\left\lbrace \boldsymbol{x}^{(n)} \right\rbrace_{n=1}^N$, we can calculate the empirical mean and covariance of the training data using:

$$
  \boldsymbol{\mu} = \frac{1}{N} \sum_{n=1}^N \left[ \boldsymbol{x}^{(n)} \right]
  \qquad
  \text{and}
  \qquad
  \mathbf{\Sigma} = \frac{1}{N} 
  \sum_{n=1}^N \left[ 
    \left(\boldsymbol{x}^{(n)} - \boldsymbol{\mu} \right)
    \left(\boldsymbol{x}^{(n)} - \boldsymbol{\mu} \right)^{\rm T}
  \right].
$$

We can then calculate an [eigendecomposition](https://en.wikipedia.org/wiki/Eigendecomposition_of_a_matrix) of the covariance matrix:

$$
  \mathbf{\Sigma} = \mathbf{Q} \mathbf{\Lambda} \mathbf{Q}^{\rm T}
  \qquad
  \mathbf{Q} = \left[ 
  \begin{array}{cccc}
  \uparrow & \uparrow & \cdots & \uparrow \\
  \boldsymbol{q}_1 & \boldsymbol{q}_2 & \cdots & \boldsymbol{q}_D \\
  \downarrow & \downarrow & \cdots & \downarrow \\
  \end{array}
  \right]
  \qquad
  \mathbf{\Lambda} = \left[ 
  \begin{array}{cccc} 
  \lambda_1 & 0 & \cdots & 0 \\
  0 & \lambda_2 & \cdots & \vdots \\
  \vdots & \vdots & \ddots & 0 \\ 
  0 & 0 & \cdots & \lambda_D \\ 
  \end{array} \right]
$$

where $\mathbf{Q}$ is an orthogonal matrix, $\mathbf{Q}\mathbf{Q}^{\rm T} = \mathbf{I}$, with columns $\left\lbrace \boldsymbol{q}_d \right\rbrace_{d=1}^D$ corresponding to the eigenvectors of $\mathbf{\Sigma}$, and $\mathbf{\Lambda}$ is a diagonal matrix with diagonal elements $\left\lbrace \lambda_d \right\rbrace_{d=1}^D$ as the corresponding eigenvalues of $\mathbf{\Sigma}$. 

Assuming the eigenvalues are ordered such that $\lambda_1 < \lambda_2 < \dots < \lambda_D$, the top $K$ principal components (eigenvectors with largest eigenvalues) correspond to $\left\lbrace \boldsymbol{q}_d \right\rbrace_{d=D + 1 - K}^D$. If we define a $D \times K$ matrix $\mathbf{V} = \left[ \boldsymbol{q}_{D + 1 - K} ~ \boldsymbol{q}_{D + 2 - K} ~\cdots~ \boldsymbol{q}_D \right]$, then we can find the projections of a (mean-normalised) input vector onto the selected $K$ principal components as $\boldsymbol{h} = \mathbf{V}^{\rm T}\left( \boldsymbol{x} - \boldsymbol{\mu}\right)$. We can then use these principal component projections to form a reconstruction of the original input using only the $K$ top principal components: $\boldsymbol{r} = \mathbf{V} \boldsymbol{h} + \boldsymbol{\mu}$. This is just a sequence of two affine transformations, directly analogous to a model with two affine layers with $K$-dimensional outputs from the first layer and inputs to the second.

---

### **Your Tasks:**

The function defined in the cell below will calculate the PCA solution for a set of input vectors and a defined number of components $K$. 

1. Use it to calculate the **top 50 principal components** of the MNIST training data. 

2. Use the returned matrix and mean vector to calculate the **PCA-based reconstructions** of a batch of 50 MNIST images.

3. Use the `show_batch_of_images` function to plot both the **original and reconstructed inputs** side by side. 

4. Calculate the **sum of squared differences error** for the PCA solution on the MNIST training set and compare it to the error from gradient descent training above. 

5. Consider: Will gradient-based training produce the same hidden representations as the PCA solution if trained to convergence?

In [ ]:
def get_pca_parameters(inputs, num_components=50):
    """Calculate PCA parameters for given inputs."""
    mean = inputs.mean(0)
    inputs_zm = inputs - mean[None, :]
    covar = np.einsum('ij,ik', inputs_zm, inputs_zm) / inputs.shape[0]
    eigvals, eigvecs = np.linalg.eigh(covar)
    return eigvecs[:, -num_components:], mean

In [ ]:
#TODO caclulate PCA reconstructions of the images

In [ ]:
#TODO calculate sum of squared differences error for PCA

---

## Exercise 2: Non-linear Autoencoders

Those who completed the extension in the previous exercise will have seen that for an autoencoder with both linear/affine encoder and decoder, there is an analytic solution for the parameters that minimise the sum of squared differences error.

In general, the advantage of using gradient-based training methods is that they allow us to use **non-linear models** for which there is no analytic solution for optimal parameters. The hope is that using non-linear transformations between affine layers will increase the **representational power** of the model. (Note: A sequence of affine transformations applied without any interleaving non-linear operations can always be represented by a single affine transformation.)

---

### **Your Tasks:**

1. Train a contractive autoencoder with the following architecture:
   - An **affine layer** (output dimension 50)
   - A **rectified linear (ReLU) layer**
   - An **affine layer** (projecting to output dimension equal to input dimension)
   - A **logistic sigmoid layer** at the output
   
2. Note: This model has the **same number of parameters** as the fully affine model in Exercise 1, since only the two affine layers have trainable parameters.

3. Train for **25 epochs** with **50 training examples per batch**.

4. Use a **uniform Glorot initialization** for weights and **zero initialization** for biases.

5. Use the **Adam** adaptive moments learning rule (`AdamLearningRule` from `mlp.learning_rules`) rather than basic gradient descent. The adaptivity helps deal with the varying scales of updates induced by non-linear transformations in this model.

In [ ]:
#TODO initialize and train the non-linear autoencoder

---


Plot batches of the inputs and reconstructed outputs for this non-linear contractive autoencoder model, and compare them to the corresponding plots for the linear models above.

In [ ]:
#TODO plot images of the original and reconstructed images

---

## Exercise 3: Denoising Autoencoders

So far, we have only considered autoencoders that try to reconstruct the input vector via an intermediate lower-dimensional "contracted" representation. The contraction is important because if we maintained the input dimensionality in all layers, a trivial optimum for the model would be to learn an identity transformation at each layer.

It can be desirable for the intermediate hidden representation to be **robust to noise** in the input. The intuition is that this will force the model to learn and maintain the "important structure" in the input (that needed to reconstruct the original input) within the hidden representation. This also removes the strict requirement for a contracted hidden representation (as the model can no longer simply learn an identity transformation). However, in practice, we will often still use a lower-dimensional hidden representation, as we believe there is redundancy in the input data and the important structure can be represented with fewer dimensions.

---

### **Your Tasks:**

1. Create a **new data provider object** that adds noise to the inputs of an autoencoder in each batch it returns. There are various ways to introduce noise. The three suggested in the lecture slides are:

   - **Gaussian noise**: Add independent, zero-mean Gaussian noise with a fixed standard deviation to each dimension of the input vectors.
   - **Masking noise**: Generate a random binary mask and perform element-wise multiplication with each input (forcing some subset of values to zero).
   - **Salt-and-pepper noise**: Select a random subset of values in each input and randomly assign either zero or one to them.
   
2. **Choose one** of these noising schemes to implement. 

3. Note: The base `DataProvider` object already has access to a random number generator as its `self.rng` attribute.

In [ ]:
class MNISTDenoisingAutoencoderDataProvider(data_providers.MNISTDataProvider):
    """Data provider for training a denoising autoencoder on MNIST."""

    def next(self):
        """Returns next data batch or raises `StopIteration` if at end."""
        return NotImplementedError("TODO implement this using a noising scheme")

---

### **Your Tasks:**

Once you have implemented your chosen noising scheme, use the new data provider object to train a **denoising autoencoder** with the **same model architecture as in Exercise 2**.

In [ ]:
#TODO train the model using the new data provider

---

Use the `show_batch_of_images` function to visualize a batch of **noisy inputs** from your data provider implementation and the **denoised reconstructions** from your trained denoising autoencoder.

In [ ]:
#TODO plot images of the original and reconstructed images

---

## Exercise 4: Using an Autoencoder as Initialization for Supervised Training

In this final exercise, we will use the first layer of an autoencoder trained on MNIST digit images as a layer within a multi-layer model trained for digit classification. 

The intuition behind pretraining methods like this is that the hidden representations learned by an autoencoder should be more useful for training a classifier than the raw pixel values. While we could fix the parameters in the layers taken from the autoencoder, we generally get better performance by allowing the whole model to be trained end-to-end on the supervised task. The learned autoencoder parameters act as a potentially more intelligent initialization than random sampling, which can help ease optimization issues caused by poor initialization.

---

### **Your Tasks:**

1. You can either:
   - Use one of the autoencoder models you trained in the previous exercises, **or**
   - Train a new autoencoder model specifically for this exercise

2. Create a **new model object** (instance of `mlp.models.MultipleLayerModel`) where:
   - The **first layer(s)** are the trained first layer(s) from your autoencoder model
   - These can be accessed via the `layers` attribute (a list of all layers in a model)
   
3. Add any additional layers you wish to the pretrained layers. At minimum, you will need to add an **output layer with dimension 10** to predict class labels.

4. Train this new model on the original MNIST image-label pairs using a **cross-entropy error**.

In [ ]:
#TODO intialize a new classification model using a trained encoder, then train the classification model

---

# PyTorch Implementation

In this section, we will construct an autoencoder using **PyTorch**. We will use the MNIST dataset with a simple linear encoder and decoder. The hidden representation will be kept relatively small (50 dimensions), and we will use the **MSE loss function** and the **Adam optimizer**. Adam is a good choice for this task as it uses adaptive learning rates, so we don't need extensive tuning.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, utils
from torch.utils.data.sampler import SubsetRandomSampler

torch.manual_seed(42)

Since we aim to encode our images in a smaller-dimensional space and then decode them back to the original space, our **output dimension should be the same as the input dimension**.

The size of `hidden_dim` is a design choice. If we choose a small `hidden_dim`, we will have a more compressed representation of our data. However, if we choose a large `hidden_dim`, we will have more accurate reconstructions, albeit with a higher risk of overfitting.

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Set training run hyperparameters
batch_size = 128  # Number of data points in a batch
learning_rate = 0.001  # Learning rate for gradient descent
num_epochs = 100  # Number of training epochs to perform
stats_interval = 5  # Epoch interval between recording and printing stats

input_dim = 1 * 28 * 28  # Images are grayscale and 28 x 28 pixels
output_dim = 1 * 28 * 28  # The output is the same size as the input image
hidden_dim = 50  # The size of the latent vector representation

For this example, we will only use the [ToTensor](https://pytorch.org/vision/main/generated/torchvision.transforms.ToTensor.html) transform. This will convert our images to PyTorch tensors and scale the pixel values to the range [0, 1].

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.MNIST('../data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('../data', train=False, download=True, transform=transform)

valid_size = 0.2  # Leave 20% of training set as validation set
num_train = len(train_dataset)
indices = list(range(num_train))
split = int(np.floor(valid_size * num_train))
np.random.shuffle(indices)  # Shuffle indices in-place
train_idx, valid_idx = indices[split:], indices[:split]  # Split indices into training and validation sets
train_sampler = SubsetRandomSampler(train_idx)
valid_sampler = SubsetRandomSampler(valid_idx)

# Create the dataloaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler, pin_memory=True)
valid_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, sampler=valid_sampler, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

We will use a **linear layer** for the encoder and a **linear layer** for the decoder. The encoder will take a 784-dimensional vector as input and output a 50-dimensional vector. The decoder will take a 50-dimensional vector as input and output a 784-dimensional vector. Although autoencoders usually have a mirror structure around the bottleneck, this is not a requirement, and you can experiment with different architectures.

We will use the **[Tanh](https://pytorch.org/docs/stable/generated/torch.nn.Tanh.html)** activation function for the encoder. Tanh is a scaled version of the sigmoid activation function and outputs values between -1 and 1. This is useful since our latent space will be normalised to have values between -1 and 1, making it easy to sample from the latent space to generate new images.

We will use the **[Sigmoid](https://pytorch.org/docs/stable/generated/torch.nn.Sigmoid.html)** activation function for the decoder. The sigmoid activation outputs values between 0 and 1.

**Question:** *Why do we want to choose the sigmoid activation for the output layer? (Think about the range of our input values.)*

Our model is composed of two `nn.Sequential` modules: one for the encoder and one for the decoder. This translates to two forward passes—one for the encoder to reduce the dimensionality of our inputs, and one for the decoder to reconstruct the original image from the latent space encoding. This is a practical choice because, after training, it allows us to use the encoder and decoder separately. For example, we can use the encoder to encode images in the latent space, then use the decoder to generate new images by sampling the latent space.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
        )

        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        x = self.decoder(z)
        return x

In [ ]:
model = Autoencoder(input_dim, output_dim, hidden_dim).to(device)
print(model)

We will use the **[MSE (Mean Squared Error)](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html)** loss function here. 

**Question:** *Why is this loss function the best choice for this task?*

In [ ]:
loss = nn.MSELoss()  # Mean squared error loss
optimizer = optim.Adam(model.parameters(), lr=learning_rate)  # Adam optimizer

Since the autoencoder is learning in an **unsupervised manner**, we no longer need the labels for training.

In [ ]:
# Keep track of the loss values over training
train_loss = [] 
valid_loss = []

# Train model
for i in range(num_epochs + 1):
    # Training
    model.train()
    batch_loss = []
    for batch_idx, (x, _) in enumerate(train_loader):
        x = x.to(device)
        x = x.view(x.size(0), -1)  # Flatten images into vectors
        
        # Forward pass
        y = model(x)
        E_value = loss(y, x)
        
        # Backward pass
        optimizer.zero_grad()
        E_value.backward()
        optimizer.step()
        
        # Logging
        batch_loss.append(E_value.item())
    
    train_loss.append(np.mean(batch_loss))

    # Validation
    model.eval()
    batch_loss = []
    for batch_idx, (x, _) in enumerate(valid_loader):
        x = x.to(device)
        x = x.view(x.size(0), -1)  # Flatten images into vectors
        
        # Forward pass
        y = model(x)
        E_value = loss(y, x)
        
        # Logging
        batch_loss.append(E_value.item())
    
    valid_loss.append(np.mean(batch_loss))

    if i % stats_interval == 0:
        print('Epoch: {} \tError(train): {:.6f} \tError(valid): {:.6f} '.format(
            i, train_loss[-1], valid_loss[-1]))

In [ ]:
# Plot the change in the validation and training set error over training
fig_1 = plt.figure(figsize=(8, 4))
ax_1 = fig_1.add_subplot(111)
ax_1.plot(train_loss, label='Error(train)')
ax_1.plot(valid_loss, label='Error(valid)')
ax_1.legend(loc=0)
ax_1.set_xlabel('Epoch number')
plt.show()

Now we can display a batch of images and their reconstructions. We can see that the reconstructions are not perfect, but they are quite good. We can also see that the reconstructions are a bit blurry. This is because we are using a linear decoder. We can improve the quality of the reconstructions by using a non-linear decoder.

In [ ]:
# Get a batch of data
x, _ = next(iter(test_loader))
x = x.to(device)
x_flatten = x.view(x.size(0), -1)

# Pass data through model
with torch.no_grad():
    y = model(x_flatten).detach().cpu().numpy()
    y = y.reshape(batch_size, 1, 28, 28)

# Plot the first ten input images and then reconstructed images
fig, axes = plt.subplots(nrows=2, ncols=10, sharex=True, sharey=True, figsize=(20, 4))

# Display input images on top row and reconstructions on bottom row
for images, row in zip([x.detach().cpu(), y], axes):
    for img, ax in zip(images, row):
        ax.imshow(np.squeeze(img), cmap='gray')
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
plt.show()

Autoencoders are **generative models** and can create *new* images by sampling the latent space. We can sample the latent space by taking a random sample from a uniform distribution on the interval $[0, 1)$ and normalising it to the interval $[-1, 1)$. We can then pass the sampled values through the decoder to generate new images.

**Question:** *Why do we need to normalise the sample in this way?*

The generative power of autoencoders is quite limited compared to other generative models, such as GANs and VAEs. However, they are much easier to train and can serve as a good starting point for more complex generative models.

**Question:** *Why is their generative power limited? (Think about the data they are trained on and the way they are trained.)*

In [ ]:
rows, cols = 2, 10
# Normalise sample to match the encoding space [-1, 1)
sample_encodings = (torch.rand(rows * cols, hidden_dim).to(device) - 0.5) * 2
with torch.no_grad():
    generated_imgs = model.decoder(sample_encodings).detach().cpu().numpy()
    generated_imgs = generated_imgs.reshape(rows * cols, 1, 28, 28)

# Plot the generated images
fig, axes = plt.subplots(nrows=2, ncols=10, sharex=True, sharey=True, figsize=(20, 4))

for images, row in zip([generated_imgs, generated_imgs[-10:]], axes):
    for img, ax in zip(images, row):
        ax.imshow(np.squeeze(img), cmap='gray')
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
plt.show()